In [1]:
#Étape 1 — Récupérer les clôtures ajustées avec Python

import yfinance as yf

tickers = ["AMD", "SOXX"]
cours = yf.download(
    tickers,
    start="2024-12-31",
    end="2026-01-01",
    interval="1d",
    auto_adjust=False,
    group_by="column",
    progress=False,
)

clotures = cours["Adj Close"][tickers].sort_index()

display(clotures.loc[["2024-12-31", "2025-12-31"]])
print("Valeurs manquantes :", clotures.isna().sum().to_dict())

Ticker,AMD,SOXX
Date,,
2024-12-31,120.790001,71.202446
2025-12-31,214.160004,100.208389


Valeurs manquantes : {'AMD': 0, 'SOXX': 0}


In [2]:
%pip install yfinance

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
Note: you may need to restart the kernel to use updated packages.


In [3]:
#Étape 2 — Calculer l’écart de rendement AMD–SOXX sur 2025
debut = clotures.loc["2024-12-31"]
fin = clotures.loc["2025-12-31"]

rendements = (fin / debut - 1) * 100
ecart = rendements["AMD"] - rendements["SOXX"]

print(f"AMD  : {rendements['AMD']:.2f} %")
print(f"SOXX : {rendements['SOXX']:.2f} %")
print(f"Écart AMD–SOXX : {ecart:+.2f} points de pourcentage")

AMD  : 77.30 %
SOXX : 40.74 %
Écart AMD–SOXX : +36.56 points de pourcentage


In [4]:
#Étape 3 — Mesurer la perte maximale depuis un sommet
sommets = clotures.cummax()
baisses = clotures / sommets - 1

perte_max = baisses.loc["2025"].min() * 100
date_du_creux = baisses.loc["2025"].idxmin()

for ticker in ["AMD", "SOXX"]:
    print(f"{ticker} : {perte_max[ticker]:.2f} % le {date_du_creux[ticker]:%d/%m/%Y}")

AMD : -39.63 % le 08/04/2025
SOXX : -34.24 % le 08/04/2025


cummax() retient, à chaque séance, la plus haute clôture observée jusque-là pour chaque titre. Le calcul inclut le 31/12/2024 comme point de départ et cherche le creux en 2025. Si le sommet est à 100 USD et que le cours retombe à 70 USD, la baisse depuis ce sommet est de −30 %.

Le 8 avril 2025, AMD était à −39,63 % de son sommet précédent, et SOXX à −34,24 % du sien. AMD a donc davantage reculé en cours d’année, tout en terminant 2025 avec un meilleur rendement. C’est une distinction que tu pourras expliquer au jury.

In [10]:
#Étape 4 — Récupérer automatiquement les chiffres du bilan AMD
import requests

from getpass import getpass

contact_sec = getpass("Adresse e-mail de contact pour la SEC : ")

cik = "0000002488"
depot = "0000002488-25-000166"
fin_trimestre = "2025-09-27"

url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
reponse = requests.get(
    url,
    headers={"User-Agent": f"Select Equity {contact_sec}"},
    timeout=30,
)
reponse.raise_for_status()
donnees = reponse.json()["facts"]["us-gaap"]

champs = {
    "Trésorerie et équivalents": "CashAndCashEquivalentsAtCarryingValue",
    "Dette à court terme": "LongTermDebtCurrent",
    "Dette à long terme": "LongTermDebtNoncurrent",
}
bilan_usd = {}
for libelle, champ in champs.items():
    montants = {
        ligne["val"]
        for ligne in donnees.get(champ, {}).get("units", {}).get("USD", [])
        if ligne["accn"] == depot and ligne["end"] == fin_trimestre
    }

    if len(montants) == 1:
        montant = montants.pop()
        bilan_usd[libelle] = montant
        print(f"{libelle} : {montant / 1_000_000:g} millions USD")
    else:
        print(f"{libelle} : à vérifier ({len(montants)} montants distincts)")

Adresse e-mail de contact pour la SEC :  ········


Trésorerie et équivalents : 4808 millions USD
Dette à court terme : 873 millions USD
Dette à long terme : 2347 millions USD


Le CIK identifie AMD, le numéro de dépôt accn identifie le 10‑Q précis et end sélectionne la date du bilan. Les deux filtres évitent de prendre un montant d’une autre période ou repris dans un rapport publié plus tard.

In [11]:
#Calculer la dette nette d'AMD
dette_nette_usd = (
    bilan_usd["Dette à court terme"]
    + bilan_usd["Dette à long terme"]
    - bilan_usd["Trésorerie et équivalents"]
)

print(f"Dette nette AMD : {dette_nette_usd / 1_000_000:.0f} millions USD")


Dette nette AMD : -1588 millions USD


Au 27 septembre 2025, la trésorerie et ses équivalents dépassaient de 1 588 millions USD la somme des deux lignes de dette retenues. C’est le chiffre du bilan de septembre que tu peux utiliser pour ton analyse au 31 décembre 2025, puisque le 10‑Q était déjà publié.

In [12]:
#Étape 5 — Calculer la marge opérationnelle d’AMD
def montant_trimestriel(champ):
    montants = {
        ligne["val"]
        for ligne in donnees[champ]["units"]["USD"]
        if ligne["accn"] == depot
        and ligne.get("start") == "2025-06-29"
        and ligne["end"] == fin_trimestre
    }

    if len(montants) != 1:
        raise ValueError(f"Vérifie le champ {champ} : {len(montants)} montants trouvés")

    return montants.pop()


chiffre_affaires = montant_trimestriel("RevenueFromContractWithCustomerExcludingAssessedTax")
resultat_operationnel = montant_trimestriel("OperatingIncomeLoss")

marge_operationnelle = resultat_operationnel / chiffre_affaires

print(f"Chiffre d’affaires : {chiffre_affaires / 1_000_000:,.0f} millions USD")
print(f"Résultat opérationnel : {resultat_operationnel / 1_000_000:,.0f} millions USD")
print(f"Marge opérationnelle AMD : {marge_operationnelle:.2%}")

Chiffre d’affaires : 9,246 millions USD
Résultat opérationnel : 1,270 millions USD
Marge opérationnelle AMD : 13.74%


Sur ce trimestre, AMD a dégagé environ 13,74 USD de résultat opérationnel pour 100 USD de ventes.

In [13]:
#Étape 6 — Récupérer les quatre bénéfices trimestriels par action
trimestres = {
    "T4 2024": ("2024-09-29", "2024-12-28"),
    "T1 2025": ("2024-12-29", "2025-03-29"),
    "T2 2025": ("2025-03-30", "2025-06-28"),
    "T3 2025": ("2025-06-29", "2025-09-27"),
}

observations = donnees["EarningsPerShareDiluted"]["units"]["USD/shares"]
bpa_trimestriels = {}

for trimestre, (debut, fin) in trimestres.items():
    candidats = [
        ligne for ligne in observations
        if ligne.get("start") == debut
        and ligne["end"] == fin
        and ligne["filed"] <= "2025-12-31"
        and ligne["form"] in ("10-K", "10-Q")
    ]

    if candidats:
        publication = min(candidats, key=lambda ligne: ligne["filed"])
        bpa_trimestriels[trimestre] = publication["val"]
        print(f"{trimestre} : {publication['val']:.2f} USD/action")
    else:
        print(f"{trimestre} : donnée trimestrielle à vérifier")

T4 2024 : donnée trimestrielle à vérifier
T1 2025 : 0.44 USD/action
T2 2025 : 0.54 USD/action
T3 2025 : 0.75 USD/action


Pour préparer le P/E historique au 31/12/2025, on récupère le bénéfice dilué par action (BPA) des quatre derniers trimestres déjà publiés à cette date. On réutilise donnees, le dictionnaire chargé depuis la SEC.

Les deux dates de période sélectionnent un trimestre seul. Le filtre sur filed limite le calcul aux chiffres connus au 31 décembre 2025. Si la SEC ne fournit pas une valeur trimestrielle séparée, le code le signale pour qu’on puisse la vérifier.

Pour le P/E au 31/12/2025, j’utilise les BPA dilués GAAP des quatre derniers
trimestres déjà publiés à cette date.

L’API SEC Company Facts a fourni les BPA des T1, T2 et T3 2025.
La recherche par dates n’a renvoyé aucune ligne pour le T4 2024.
Pour ce test pilote, j’ai donc ajouté manuellement le BPA dilué GAAP
de 0,29 USD par action, vérifié dans le communiqué d’AMD publié
le 4 février 2025 et joint au formulaire 8-K.

[Source SEC — communiqué AMD, annexe 99.1](https://www.sec.gov/Archives/edgar/data/2488/000000248825000009/q42024991.htm)

In [14]:
bpa_trimestriels["T4 2024"] = 0.29
print(f"T4 2024 : {bpa_trimestriels['T4 2024']:.2f} USD/action")
bpa_12_mois = sum(bpa_trimestriels[trimestre] for trimestre in trimestres)
cours_amd = cours["Close"]["AMD"].loc["2025-12-31"]
pe_amd = cours_amd / bpa_12_mois

print(f"BPA AMD sur quatre trimestres : {bpa_12_mois:.2f} USD/action")
print(f"P/E AMD au 31/12/2025 : {pe_amd:.2f}×")

T4 2024 : 0.29 USD/action
BPA AMD sur quatre trimestres : 2.02 USD/action
P/E AMD au 31/12/2025 : 106.02×


Les quatre BPA dilués GAAP totalisent 2,02 USD par action. En divisant le cours de clôture d’AMD au 31 décembre 2025 (214,16 USD) par ce montant, tu obtiens bien un P/E historique de 106,02×.

#Étape 7 — Résumé du test pilote 
### Synthèse du test pilote AMD–SOXX au 31/12/2025

| Indicateur | Résultat AMD | Période ou référence |
|---|---:|---|
| Écart de rendement vs SOXX | +36,56 points de pourcentage | 2025. AMD +77,30 %, SOXX +40,74 % |
| Perte maximale depuis un sommet | −39,63 % | Creux le 08/04/2025. SOXX −34,24 % |
| Dette nette | −1 588 millions USD | Bilan au 27/09/2025, publié le 05/11/2025 |
| Marge opérationnelle | 13,74 % | Trimestre clos le 27/09/2025 |
| P/E historique | 106,02× | Cours au 31/12/2025 et BPA dilué GAAP des quatre derniers trimestres publiés |